In [1]:
import genetick_AI
import pandas
import numpy as np

def f_2(x):
    return ((1 - x) ** 76)/((1 - x) ** 76 + 10 * x ** 12)

def f_3(x):
    return (1-x)*(1-f_2(x)-f_5(x))

def f_4(x):
    return f_3(1 - x)

def f_5(x):
    return f_2(1 - x)

In [37]:
n,m = 100,100
matrix, perf, diff = genetick_AI.generate_rating_matrix(n,m)
df = pandas.DataFrame(matrix)
df

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,4.0,4.0,4.0,4.0,4.0,3.0,3.0,3.0,3.0,3.0,...,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,4.0
1,3.0,3.0,4.0,3.0,3.0,4.0,4.0,4.0,4.0,4.0,...,3.0,4.0,4.0,4.0,4.0,3.0,3.0,3.0,3.0,3.0
2,3.0,3.0,4.0,4.0,4.0,3.0,4.0,3.0,3.0,4.0,...,3.0,3.0,4.0,4.0,3.0,3.0,4.0,3.0,4.0,3.0
3,3.0,4.0,3.0,3.0,4.0,3.0,3.0,4.0,4.0,4.0,...,4.0,4.0,3.0,4.0,4.0,4.0,4.0,3.0,4.0,4.0
4,4.0,3.0,4.0,3.0,4.0,3.0,3.0,4.0,4.0,3.0,...,4.0,4.0,4.0,4.0,4.0,3.0,3.0,3.0,4.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,3.0,4.0,4.0,4.0,4.0,3.0,3.0,3.0,4.0,3.0,...,3.0,4.0,4.0,4.0,3.0,4.0,4.0,3.0,4.0,4.0
96,4.0,3.0,4.0,3.0,4.0,4.0,4.0,3.0,4.0,3.0,...,4.0,4.0,3.0,3.0,3.0,4.0,4.0,4.0,4.0,4.0
97,4.0,4.0,4.0,4.0,3.0,3.0,3.0,3.0,3.0,4.0,...,3.0,4.0,4.0,4.0,4.0,3.0,4.0,3.0,4.0,4.0
98,3.0,4.0,3.0,3.0,3.0,3.0,4.0,4.0,4.0,4.0,...,3.0,4.0,3.0,3.0,4.0,3.0,4.0,4.0,4.0,4.0


In [38]:
def calculate_f_matrix(ratings_matrix, st_it, student_alpha=2, student_beta=2,
                       item_alpha=2, item_beta=2):
    """
    Returns a matrix of f_rating contributions (without Beta priors) for observed ratings.

    Parameters
    ----------
    ratings_matrix : np.ndarray, shape (n_students, n_items)
        Observed ratings (2..5). Unobserved entries should be 0, NaN, or a sentinel.
    st_it : np.ndarray
        Concatenated vector: first n_students entries = student parameters,
        remaining = item parameters. All values in (0,1).
    student_alpha, student_beta, item_alpha, item_beta : float
        Ignored (kept only for API compatibility). Excluded from calculation.

    Returns
    -------
    f_matrix : np.ndarray, same shape as ratings_matrix
        For each observed rating, f_rating( student_i / (student_i + item_j) ).
        Unobserved cells are set to 0.0.
    """
    # Dictionary mapping rating values to their corresponding functions
    methods = {2: f_2, 3: f_3, 4: f_4, 5: f_5}   # f_2..f_5 must be defined elsewhere

    n_students, n_items = ratings_matrix.shape
    students = st_it[:n_students]
    items = st_it[n_students:]

    # Initialize matrix with zeros (unrated cells stay 0)
    f_matrix = np.zeros_like(ratings_matrix, dtype=float)

    # Fill f_rating contributions for each observed rating value
    for rating, func in methods.items():
        mask = (ratings_matrix == rating)
        if not np.any(mask):
            continue
        rows, cols = np.where(mask)
        v1_vals = students[rows]
        v2_vals = items[cols]
        ratio = v1_vals / (v1_vals + v2_vals)
        f_vals = func(ratio)
        f_matrix[rows, cols] = f_vals

    return f_matrix
a = np.concatenate((perf,diff))
prob_m = calculate_f_matrix(matrix,a)
df = pandas.DataFrame(prob_m)
df

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,0.629259,0.558056,0.611921,0.477348,0.526221,0.540209,0.397262,0.467122,0.402576,0.588269,...,0.495815,0.274989,0.608872,0.352894,0.422031,0.572488,0.335917,0.461810,0.291818,0.481422
1,0.420436,0.493695,0.561521,0.574131,0.525746,0.408726,0.552020,0.480921,0.546538,0.362421,...,0.547683,0.681655,0.342849,0.598277,0.526572,0.622475,0.383791,0.513746,0.336592,0.570134
2,0.369619,0.440756,0.613064,0.478549,0.527421,0.539012,0.603891,0.465924,0.401418,0.412898,...,0.494611,0.274029,0.392276,0.648205,0.420857,0.571309,0.665157,0.460613,0.709176,0.517375
3,0.535906,0.391828,0.554166,0.682131,0.361715,0.697219,0.563658,0.367906,0.430903,0.263117,...,0.341601,0.573594,0.752504,0.483365,0.411330,0.275891,0.502158,0.627113,0.553213,0.321420
4,0.713319,0.350736,0.698028,0.427548,0.619520,0.444888,0.310150,0.625798,0.685093,0.493571,...,0.598513,0.549090,0.484994,0.728849,0.667516,0.477385,0.256480,0.369214,0.740891,0.423557
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,0.394602,0.533014,0.587675,0.452221,0.500988,0.565180,0.421685,0.492333,0.572905,0.612503,...,0.521059,0.704414,0.367350,0.623708,0.446848,0.402986,0.641186,0.486996,0.686873,0.456269
96,0.654107,0.415474,0.637261,0.495642,0.553070,0.486734,0.628314,0.440335,0.623128,0.561859,...,0.531171,0.745808,0.582846,0.328617,0.395908,0.454152,0.687756,0.564922,0.729990,0.508439
97,0.654724,0.585188,0.637891,0.505040,0.446256,0.512585,0.371049,0.439663,0.376231,0.438812,...,0.468150,0.746306,0.417817,0.671985,0.604744,0.545171,0.688341,0.434408,0.730526,0.509121
98,0.346269,0.584122,0.363123,0.496057,0.447340,0.513681,0.627927,0.559256,0.622738,0.437732,...,0.469243,0.745503,0.583250,0.328983,0.603695,0.546259,0.687399,0.564514,0.729664,0.508024


In [39]:
del a
res = genetick_AI.try_find_opt(matrix)

\\?\C:\Users\user\AppData\Roaming\jupyterlab-desktop\jlab_server\Lib\site-packages\deap\creator.py:185: RuntimeWarning: A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
\\?\C:\Users\user\AppData\Roaming\jupyterlab-desktop\jlab_server\Lib\site-packages\deap\creator.py:185: RuntimeWarning: A class named 'Individual' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "


In [40]:
# print(pandas.Series(res[:n]),'\n',pandas.Series(res[n:]))
# print(pandas.Series(perf),'\n', pandas.Series(diff))
df1 = pandas.DataFrame((pandas.Series(res[:n]),pandas.Series(perf)))
df2 = pandas.DataFrame((pandas.Series(res[n:]),pandas.Series(diff)))
df1

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,0.615635,0.499997,0.618608,0.31411,0.902516,0.241679,0.758234,0.368389,0.432748,0.720396,...,0.635817,0.501903,0.701662,0.585085,0.118386,0.556477,0.685919,0.687792,0.684781,0.646766
1,0.615635,0.499997,0.618608,0.31411,0.902516,0.241679,0.758234,0.368389,0.432748,0.720396,...,0.635817,0.501903,0.701662,0.585085,0.118386,0.556477,0.685919,0.687792,0.684781,0.646766


In [41]:
df2

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,0.362715,0.487543,0.390435,0.674065,0.554283,0.72331,0.405762,0.539668,0.414847,0.879603,...,0.605415,0.233508,0.958363,0.335731,0.449535,0.824409,0.31141,0.528265,0.253683,0.663149
1,0.362715,0.487543,0.390435,0.674065,0.554283,0.72331,0.405762,0.539668,0.414847,0.879603,...,0.605415,0.233508,0.958363,0.335731,0.449535,0.824409,0.31141,0.528265,0.253683,0.663149


In [42]:
print(np.abs(perf-res[:n]),'\n',np.abs(diff-res[n:]))
print(max(np.abs(perf-res[:n])),max(np.abs(diff-res[n:])))

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0.] 
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0.]
0.0 0.0


In [34]:
prob_m_ap = calculate_f_matrix(matrix,np.array(res))
df1 = pandas.DataFrame(prob_m_ap)
df1

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,0.579184,0.241469,0.492385,0.533699,0.712005,0.492119,0.698476,0.544310,0.382617,0.415763,...,0.370741,0.441944,0.388079,0.522652,0.473779,0.459791,0.602738,0.467122,0.597424,0.588269
1,0.628894,0.717927,0.544281,0.481745,0.332456,0.559611,0.652943,0.492414,0.432808,0.532986,...,0.579564,0.506305,0.438479,0.425869,0.525746,0.591274,0.552020,0.519079,0.546538,0.637579
2,0.578009,0.240527,0.491181,0.465103,0.712991,0.493322,0.699489,0.454496,0.381480,0.585406,...,0.369619,0.559244,0.386936,0.521451,0.527421,0.539012,0.396109,0.465924,0.401418,0.412898
3,0.270445,0.615232,0.655307,0.631326,0.557800,0.669171,0.541689,0.378668,0.548461,0.582421,...,0.464094,0.608172,0.445834,0.317869,0.361715,0.302780,0.563658,0.632094,0.569097,0.736807
4,0.515772,0.967430,0.398194,0.626570,0.721186,0.586861,0.760528,0.363493,0.297133,0.673206,...,0.713319,0.649264,0.698028,0.572452,0.380480,0.555112,0.689850,0.625798,0.685093,0.493571
5,0.750632,0.551622,0.711888,0.689984,0.507473,0.724427,0.476270,0.319225,0.612204,0.355522,...,0.399870,0.668579,0.617665,0.263898,0.303632,0.749176,0.373284,0.309311,0.631882,0.715332
6,0.472257,0.554710,0.440582,0.585001,0.247021,0.544088,0.259501,0.595330,0.334747,0.366206,...,0.676422,0.608644,0.339902,0.470617,0.577695,0.511787,0.348594,0.584200,0.646361,0.462948
7,0.696976,0.652208,0.381532,0.593515,0.403324,0.367015,0.580917,0.416828,0.491236,0.543223,...,0.496119,0.569605,0.514526,0.646613,0.600736,0.337446,0.475862,0.594310,0.529658,0.295185
8,0.661934,0.312218,0.579821,0.445838,0.634750,0.594844,0.380470,0.456412,0.468553,0.503077,...,0.544020,0.529771,0.525700,0.609014,0.438434,0.374331,0.516092,0.554977,0.489440,0.670250
9,0.459519,0.703180,0.453235,0.572521,0.742986,0.468637,0.730486,0.582939,0.653758,0.621833,...,0.334883,0.403616,0.648519,0.516613,0.565159,0.501009,0.639694,0.428286,0.634574,0.450248


In [35]:
df

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,0.579184,0.241469,0.492385,0.533699,0.712005,0.492119,0.698476,0.544310,0.382617,0.415763,...,0.370741,0.441944,0.388079,0.522652,0.473779,0.459791,0.602738,0.467122,0.597424,0.588269
1,0.628894,0.717927,0.544281,0.481745,0.332456,0.559611,0.652943,0.492414,0.432808,0.532986,...,0.579564,0.506305,0.438479,0.425869,0.525746,0.591274,0.552020,0.519079,0.546538,0.637579
2,0.578009,0.240527,0.491181,0.465103,0.712991,0.493322,0.699489,0.454496,0.381480,0.585406,...,0.369619,0.559244,0.386936,0.521451,0.527421,0.539012,0.396109,0.465924,0.401418,0.412898
3,0.270445,0.615232,0.655307,0.631326,0.557800,0.669171,0.541689,0.378668,0.548461,0.582421,...,0.464094,0.608172,0.445834,0.317869,0.361715,0.302780,0.563658,0.632094,0.569097,0.736807
4,0.515772,0.967430,0.398194,0.626570,0.721186,0.586861,0.760528,0.363493,0.297133,0.673206,...,0.713319,0.649264,0.698028,0.572452,0.380480,0.555112,0.689850,0.625798,0.685093,0.493571
5,0.750632,0.551622,0.711888,0.689984,0.507473,0.724427,0.476270,0.319225,0.612204,0.355522,...,0.399870,0.668579,0.617665,0.263898,0.303632,0.749176,0.373284,0.309311,0.631882,0.715332
6,0.472257,0.554710,0.440582,0.585001,0.247021,0.544088,0.259501,0.595330,0.334747,0.366206,...,0.676422,0.608644,0.339902,0.470617,0.577695,0.511787,0.348594,0.584200,0.646361,0.462948
7,0.696976,0.652208,0.381532,0.593515,0.403324,0.367015,0.580917,0.416828,0.491236,0.543223,...,0.496119,0.569605,0.514526,0.646613,0.600736,0.337446,0.475862,0.594310,0.529658,0.295185
8,0.661934,0.312218,0.579821,0.445838,0.634750,0.594844,0.380470,0.456412,0.468553,0.503077,...,0.544020,0.529771,0.525700,0.609014,0.438434,0.374331,0.516092,0.554977,0.489440,0.670250
9,0.459519,0.703180,0.453235,0.572521,0.742986,0.468637,0.730486,0.582939,0.653758,0.621833,...,0.334883,0.403616,0.648519,0.516613,0.565159,0.501009,0.639694,0.428286,0.634574,0.450248


In [36]:
print(df.prod().prod(),df1.prod().prod(),(df1/df).prod().prod(),df1.prod().prod()/df.prod().prod())

2.8168521992281023e-279 2.8168521992281023e-279 1.0 1.0
